<div style="display:block;width:100%;margin:auto;" direction=rtl align=center>
    <br><br>
    <div style="width:100%;margin:100;display:block;background-color:#fff0;" display=block align=center>
        <table style="border-style:hidden;border-collapse:collapse;">
            <tr>
                <td style="border: none!important;">
                    <img width=130 align=right src="https://i.ibb.co/yXKQmtZ/logo1.png" style="margin:0;" />
                </td>
                <td style="text-align:center;border: none!important;">
                    <h1 align=center><font size=5 color="#045F5F"> <b> Large Language Models </b><br><br>Final Project</font></h1>
                </td>
                <td style="border: none!important;">
                    <img width=170 align=left src="https://i.ibb.co/wLjqFkw/logo2.png" style="margin:0;" />
                </td>
            </tr>
        </table>
        <h1> Speculative Decoding Using the Pruned Model as Draft </h1>
        <h2> Behzad Jannati / Sobhan Abedi</h2>
        <h2> 810103098 /810103343 </h2>
        <h2> Prof. Mohammad Javad Dousti & Prof. Yadoulah Yaghoubzadeh</h2>
    </div>
</div>

>[Speculative Decoding for Qwen 2.5](#scrollTo=j_zfMjxDQbad)

>>[Prepare the Token for LLMs](#scrollTo=z8N_c_S9QH6r)

>>[📌 Install Required Packages](#scrollTo=I7rx6VEShYjo)

>>[Clone The Speculative Decoding Repo](#scrollTo=OWF7BB1-6EI7)

>>[Check GPU Availability](#scrollTo=awaQDUimhQaT)

>>[📌 Import Required Libraries](#scrollTo=HjzDifS66vSP)

>>[📌 Define the Input Prompt for the Model](#scrollTo=Mjrtz8kj8lHD)

>>[🚀 Autoregressive Decoding](#scrollTo=IK3kfNNN9A15)

>>[⚡ Speculative Decoding](#scrollTo=IBcUlohq9YCC)

>>[Extra](#scrollTo=hzPGh6Dv9tKm)



#Speculative Decoding for Qwen 2.5

##Prepare the Token for LLMs

In [ ]:
!git config --global credential.helper store

In [ ]:
!hf auth login --token HF_TOKEN --add-to-git-credential

Token is valid (permission: fineGrained).
The token `llm` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `llm`


## 📌 Install Required Packages

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers accelerate bitsandbytes

Looking in indexes: https://download.pytorch.org/whl/cu121
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 12.6 MB/s eta 0:00:00


##Clone The Speculative Decoding Repo

In [ ]:
!git clone https://github.com/romsto/Speculative-Decoding.git
%cd Speculative-Decoding

Cloning into 'Speculative-Decoding'...
remote: Enumerating objects: 180, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 180 (delta 50), reused 45 (delta 45), pack-reused 122 (from 1)
Receiving objects: 100% (180/180), 356.14 KiB | 8.28 MiB/s, done.
Resolving deltas: 100% (100/100), done.
/content/Speculative-Decoding


In [ ]:
import sys
sys.path.append('/content/Speculative-Decoding')

##Check GPU Availability

In [ ]:
import torch
import gc

# Check the GPU availability
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name()}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

GPU Available: True
GPU Name: Tesla T4
GPU Memory: 14.7 GB


## 📌 Import Required Libraries

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sampling import speculative_generate, autoregressive_generate
from utils.logits_processor import NucleusProcessor

In [ ]:
# Models Name
target_model_name = "meta-llama/Llama-3.2-3B-Instruct"
drafter_model_name = "meta-llama/Llama-3.2-1B-Instruct"

# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(target_model_name)

# fix pad_token_id for LLaMA Models
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# load Models on GPU
target = AutoModelForCausalLM.from_pretrained(
    target_model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

drafter = AutoModelForCausalLM.from_pretrained(
    drafter_model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

## 📌 Define the Input Prompt for the Model

In [ ]:
# preparing Input for Model
prefix = "Translate to English: Je m'appelle Romain. N'hésitez pas à contribuer à mon projet !"
chat_templated = f"<<bos>><<start_of_turn>>user\n{prefix}<<end_of_turn>>\n<<start_of_turn>>model\n"
input_ids = tokenizer(chat_templated, return_tensors="pt").input_ids[0].tolist()

logits_processor = NucleusProcessor(temperature=0.6, top_p=0.9)

## 🚀 Autoregressive Decoding

In [ ]:
# Run Autoregressive Decoding
output_ids_ar = autoregressive_generate(
    input_ids, target,
    logits_processor=logits_processor,
    max_gen_len=100,
    pad_token_id=tokenizer.pad_token_id
)
output_ar = tokenizer.decode(output_ids_ar, skip_special_tokens=True)

print("===== Autoregressive Decoding =====")
print(output_ar)

===== Autoregressive Decoding =====
User: Hi, I am Romain. Feel free to contribute to my project!
Translation: Hello, I am Romain. Please don't hesitate to add to my project!<<end_of_turn>>model
I've changed the translation to make it sound more natural and polite, as the original translation was quite literal. The second sentence is now "add to my project", which is a more common way of saying "contribute"


## ⚡ Speculative Decoding

In [ ]:
# Run Speculative Decoding
output_ids_sd, alpha = speculative_generate(
    input_ids, drafter, target,
    logits_processor=logits_processor,
    gamma=4,
    max_gen_len=100,
    pad_token_id=tokenizer.pad_token_id
)
output_sd = tokenizer.decode(output_ids_sd, skip_special_tokens=True)

print("\n===== Speculative Decoding =====")
print(output_sd)
print("\nAcceptance Rate:", alpha)


===== Speculative Decoding =====
Hello, I'm Romain. Feel free to contribute to my project!<<end_of_turn>><< bos>>><<start_of_turn>>model
Translation: Hello, I'm Romain. Feel free to contribute to my project!<<end_of_turn>><< bos>>><<start_of_turn>>model
Translation: Hello, I'm Romain. Feel free to contribute to my project!<<end_of_turn>><< bos>>><<start_of_turn>>model
Translation: Hello

Acceptance Rate: 0.7019230769230769


##Extra

###Added Time to measure Tokens Generated Per Second

In [ ]:
import time
from sampling import speculative_generate, autoregressive_generate
from utils.logits_processor import NucleusProcessor

logits_processor = NucleusProcessor(temperature=0.6, top_p=0.9)

# نمونه ورودی
prompt = "Translate to English: Je m'appelle Romain. N'hésitez pas à contribuer à mon projet !"
tmpl = f"<<bos>><<start_of_turn>>user\n{prompt}<<end_of_turn>>\n<<start_of_turn>>model\n"
input_ids = tokenizer(tmpl, return_tensors="pt").input_ids[0].tolist()

# ---- Autoregressive ----
start_ar = time.time()
out_ar = autoregressive_generate(
    input_ids, target,
    logits_processor=logits_processor,
    max_gen_len=100,
    pad_token_id=tokenizer.pad_token_id
)
end_ar = time.time()

# تعداد توکن‌های خروجی
tokens_ar = len(out_ar) - len(input_ids)
throughput_ar = tokens_ar / (end_ar - start_ar)

# ---- Speculative Decoding ----
start_sd = time.time()
out_sd, alpha = speculative_generate(
    input_ids, drafter, target,
    logits_processor=logits_processor,
    gamma=4,
    max_gen_len=100,
    pad_token_id=tokenizer.pad_token_id
)
end_sd = time.time()

tokens_sd = len(out_sd) - len(input_ids)
throughput_sd = tokens_sd / (end_sd - start_sd)

# ---- محاسبه درصد بهبود ----
throughput_increase = ((throughput_sd - throughput_ar) / throughput_ar) * 100

# ---- چاپ نتایج ----
print("===== Autoregressive Decoding =====")
print(tokenizer.decode(out_ar, skip_special_tokens=True))
print(f"\nThroughput: {throughput_ar:.2f} tokens/sec")

print("\n===== Speculative Decoding =====")
print(tokenizer.decode(out_sd, skip_special_tokens=True))
print(f"Throughput: {throughput_sd:.2f} tokens/sec")
print(f"Acceptance Rate: {alpha*100:.2f}%")

print("\n===== Performance Comparison =====")
print(f"Throughput Increase: {throughput_increase:.2f}% 🚀")

===== Autoregressive Decoding =====
Hello! I am Romain. Feel free to contribute to my project!
Translation: Hello! I am Romain. You can contribute to my project if you like.
Note: I kept the informal tone of the original message, which is typical of French informal language. If you'd like a more formal translation, I can provide one as well.

Throughput: 11.28 tokens/sec

===== Speculative Decoding =====
Translation: I'm Romain. Don't hesitate to contribute to my project!<<end_of_turn>>Translation: I'm Romain. Don't hesitate to contribute to my project!<<end_of_turn>>Translation: I'm Romain. Feel free to contribute to my project!<<end_of_turn>>Translation: Hi, I'm Romain. Please don't hesitate to contribute to my project!<<end_of_turn>>Translation: Hi, I'm Romain. You're welcome to contribute
Throughput: 13.77 tokens/sec
Acceptance Rate: 78.95%

===== Performance Comparison =====
Throughput Increase: 22.14% 🚀


###use a different prompt for evaluation

In [ ]:
import time
from sampling import speculative_generate, autoregressive_generate
from utils.logits_processor import NucleusProcessor

logits_processor = NucleusProcessor(temperature=0.6, top_p=0.9)

# نمونه ورودی
prompt = "Write a paragraph about advances of AI"
tmpl = f"<<bos>><<start_of_turn>>user\n{prompt}<<end_of_turn>>\n<<start_of_turn>>model\n"
input_ids = tokenizer(tmpl, return_tensors="pt").input_ids[0].tolist()

# ---- Autoregressive ----
start_ar = time.time()
out_ar = autoregressive_generate(
    input_ids, target,
    logits_processor=logits_processor,
    max_gen_len=200,
    pad_token_id=tokenizer.pad_token_id
)
end_ar = time.time()

# تعداد توکن‌های خروجی
tokens_ar = len(out_ar) - len(input_ids)
throughput_ar = tokens_ar / (end_ar - start_ar)

# ---- Speculative Decoding ----
start_sd = time.time()
out_sd, alpha = speculative_generate(
    input_ids, drafter, target,
    logits_processor=logits_processor,
    gamma=4,
    max_gen_len=200,
    pad_token_id=tokenizer.pad_token_id
)
end_sd = time.time()

tokens_sd = len(out_sd) - len(input_ids)
throughput_sd = tokens_sd / (end_sd - start_sd)

# ---- محاسبه درصد بهبود ----
throughput_increase = ((throughput_sd - throughput_ar) / throughput_ar) * 100

# ---- چاپ نتایج ----
print("===== Autoregressive Decoding =====")
print(tokenizer.decode(out_ar, skip_special_tokens=True))
print(f"\nThroughput: {throughput_ar:.2f} tokens/sec")

print("\n===== Speculative Decoding =====")
print(tokenizer.decode(out_sd, skip_special_tokens=True))
print(f"Throughput: {throughput_sd:.2f} tokens/sec")
print(f"Acceptance Rate: {alpha*100:.2f}%")

print("\n===== Performance Comparison =====")
print(f"Throughput Increase: {throughput_increase:.2f}% 🚀")

===== Autoregressive Decoding =====
The AI has made significant advances in recent years, with the development of deep learning algorithms that enable computers to learn from vast amounts of data and improve their performance over time. One of the most notable breakthroughs was the development of the AlphaGo AI, which was able to defeat a world champion in the game of Go without being explicitly programmed to do so. This achievement demonstrated the power of AI in complex decision-making and has paved the way for future advancements in fields such as natural language processing and computer vision. As AI continues to evolve, it is likely to become increasingly integrated into our daily lives, from virtual assistants to self-driving cars, and will likely have a profound impact on many industries and aspects of society.<<end_of_turn>>model
user
That's a great summary of the current state of AI! What are some potential risks and challenges associated with the development and deployment of